# Simple forecasting and recommendation baselines

Run the cells from top to bottom using the project's Python environment. These rules do **not** need model fitting. We evaluate them, compare their errors, and record each experiment in MLflow.

You will learn how to:
1. Predict tomorrow with the last observed day, weekly/28-day seasonal naive, and a seven-day moving average.
2. Recommend products using global popularity, category popularity, and purchase history.
3. Evaluate on chronological validation/test periods and log parameters, metrics and summaries.

Prerequisites: `scripts/prepare_curated_data.py` and `scripts/build_forecasting_features.py` have produced the warehouse and feature Parquet files. Select `.venv` as your notebook environment. If it has no notebook kernel, install `ipykernel` in that environment first. No data preparation files are modified by this notebook.

In [1]:
from pathlib import Path
import json
import math
import os
import sys

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
import duckdb
import numpy as np
import pandas as pd
import mlflow
from IPython.display import display

# Works when the notebook starts in the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/build_forecasting_features.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start this notebook inside the project directory")
sys.path.insert(0, str(ROOT))
from scripts.build_recommendation_features import load_history

FEATURES = ROOT / "data/features/forecasting"
WAREHOUSE = ROOT / "data/warehouse/olist.duckdb"
OUTPUT = ROOT / "data/baselines"
OUTPUT.mkdir(parents=True, exist_ok=True)
for path in [WAREHOUSE, FEATURES / "metadata.json", *[FEATURES / f"{s}.parquet" for s in ("train", "validation", "test")]]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing prerequisite: {path}")

# Always use a local store; this does not contact a remote tracking server.
mlflow.set_tracking_uri(f"sqlite:///{OUTPUT / 'mlflow.db'}")
mlflow.set_experiment("simple-ecommerce-baselines")
print("Repository:", ROOT)
print("MLflow store:", mlflow.get_tracking_uri())

/Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repository: /Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender
MLflow store: sqlite:////Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/data/baselines/mlflow.db


## 1. Understand the forecasting dates

A feature row dated **January 10** predicts demand on **January 11**. We assume January 10 demand is already known.

| Rule | What we read for a January 10 feature row |
|---|---|
| Last-day | January 10 demand |
| Weekly seasonal naive / same-day-last-week | January 4 demand (six rows before January 10) |
| Seven-day moving average | January 4–10, including the current row |
| 28-day seasonal naive | December 14 demand (27 rows before January 10) |

Weekly seasonal naive and same-day-last-week are the **same baseline**, so we evaluate them once. The existing `demand_lag_7` and `rolling_mean_7` are offset differently: the latter excludes today. We recompute precisely aligned values below.

In [2]:
toy = pd.DataFrame({"purchase_date": pd.date_range("2025-01-01", periods=10),
                    "demand": [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]})
toy["forecast_date"] = toy.purchase_date + pd.Timedelta(days=1)
toy["last_day"] = toy.demand
toy["weekly_seasonal_naive"] = toy.demand.shift(6)
toy["moving_average_7"] = toy.demand.rolling(7, min_periods=7).mean()
display(toy.tail(4))
# January 10 -> January 11: use Jan 4 for weekly and Jan 4–10 for average.
assert toy.iloc[-1].weekly_seasonal_naive == 5
assert toy.iloc[-1].moving_average_7 == 8

,purchase_date,demand,forecast_date,last_day,weekly_seasonal_naive,moving_average_7
6,2025-01-07,8,2025-01-08,8,2.0,5.0
7,2025-01-08,9,2025-01-09,9,3.0,6.0
8,2025-01-09,10,2025-01-10,10,4.0,7.0
9,2025-01-10,11,2025-01-11,11,5.0,8.0


## 2. Load evaluation boundaries without loading the full training table

DuckDB reads just the necessary columns and dates from Parquet. We include 28 days of training context before validation so the first evaluated rows have historical values. Window functions partition by product/state and order by date, exactly as in the feature-building script.

This is a **daily rolling one-step evaluation**: when predicting tomorrow, today's actual demand is available. It is not a fixed-origin forecast of all 30 future days.

In [3]:
metadata = json.loads((FEATURES / "metadata.json").read_text())
ranges = metadata["date_ranges"]
validation_start = pd.Timestamp(ranges["validation"]["min"])
test_start = pd.Timestamp(ranges["test"]["min"])
context_start = validation_start - pd.Timedelta(days=28)

def sql_path(path):
    return str(path).replace("'", "''")

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=2")
con.execute(f"SET temp_directory='{sql_path(OUTPUT / 'duckdb_tmp')}'")
parts = []
for split in ("train", "validation", "test"):
    parts.append(f"""SELECT purchase_date, product_id, customer_state,
                           CAST(demand AS DOUBLE) AS demand,
                           CAST(target_demand_next_day AS DOUBLE) AS actual,
                           '{split}' AS split
                    FROM read_parquet('{sql_path(FEATURES / (split + '.parquet'))}')
                    WHERE purchase_date >= DATE '{context_start.date()}'""")
con.execute("CREATE OR REPLACE TEMP VIEW evaluation_input AS " + " UNION ALL ".join(parts))
con.execute("""
    CREATE OR REPLACE TEMP TABLE forecast_predictions AS
    WITH calculated AS (
        SELECT *, demand AS last_day,
               LAG(demand, 6) OVER series AS weekly_seasonal_naive,
               LAG(demand, 27) OVER series AS seasonal_naive_28,
               CASE WHEN COUNT(*) OVER recent = 7
                    THEN AVG(demand) OVER recent END AS moving_average_7
        FROM evaluation_input
        WINDOW series AS (PARTITION BY product_id, customer_state ORDER BY purchase_date),
               recent AS (PARTITION BY product_id, customer_state ORDER BY purchase_date
                          ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)
    )
    SELECT * FROM calculated WHERE split IN ('validation', 'test')
""")
coverage = con.sql("""SELECT split, COUNT(*) AS rows,
    COUNT(*) FILTER (WHERE weekly_seasonal_naive IS NULL OR seasonal_naive_28 IS NULL
                           OR moving_average_7 IS NULL OR actual IS NULL) AS incomplete_rows,
    MIN(purchase_date) AS first_feature_date, MAX(purchase_date) AS last_feature_date
    FROM forecast_predictions GROUP BY split ORDER BY split""").df()
display(coverage)
assert coverage.incomplete_rows.sum() == 0, "Incomplete history: check split continuity"
# Confirm dense daily spacing for evaluated histories; row lags must represent days.
assert con.sql("""WITH gaps AS (
    SELECT purchase_date, split,
           purchase_date - LAG(purchase_date) OVER (
               PARTITION BY product_id, customer_state ORDER BY purchase_date) AS gap
    FROM evaluation_input)
    SELECT COUNT(*) FROM gaps WHERE split != 'train' AND gap != INTERVAL 1 DAY
""").fetchone()[0] == 0

,split,rows,incomplete_rows,first_feature_date,last_feature_date
0,test,1836720,0,2018-08-04,2018-09-02
1,validation,1836720,0,2018-07-05,2018-08-03


## 3. Measure forecasting error

- **MAE:** average absolute miss, in items. Lower is better.
- **RMSE:** gives larger misses more weight. Lower is better.
- **WAPE:** total absolute error divided by total actual demand. Undefined when total demand is zero.
- **Positive-demand MAE:** error only on rows where actual demand is positive.

We also include an `always_zero` diagnostic. The dense product/state table contains many zeros, so a low overall MAE alone can be misleading. Predictions remain fractional for averages; we do not round them before evaluation.

In [4]:
FORECAST_METHODS = {
    "last_day": "last_day",
    "weekly_seasonal_naive": "weekly_seasonal_naive",
    "moving_average_7": "moving_average_7",
    "seasonal_naive_28": "seasonal_naive_28",
    "always_zero": "0.0",
}
forecast_rows = []
for method, expression in FORECAST_METHODS.items():
    frame = con.sql(f"""
        SELECT split, COUNT(*) AS rows,
               AVG(ABS(actual - ({expression}))) AS mae,
               SQRT(AVG(POWER(actual - ({expression}), 2))) AS rmse,
               SUM(ABS(actual - ({expression}))) / NULLIF(SUM(actual), 0) AS wape,
               AVG(ABS(actual - ({expression}))) FILTER (WHERE actual > 0) AS positive_demand_mae,
               AVG(CASE WHEN actual = 0 THEN 1.0 ELSE 0.0 END) AS zero_demand_fraction
        FROM forecast_predictions GROUP BY split
    """).df()
    frame["baseline"] = method
    forecast_rows.append(frame)
forecast_metrics = pd.concat(forecast_rows, ignore_index=True)
display(forecast_metrics.sort_values(["split", "mae"]).reset_index(drop=True))
forecast_metrics.to_csv(OUTPUT / "forecast_metrics.csv", index=False)
forecast_winner = (forecast_metrics.query("split == 'validation'")
                   .sort_values(["mae", "baseline"]).iloc[0].baseline)
print("Selected on validation MAE:", forecast_winner)
display(forecast_metrics.query("split == 'test' and baseline == @forecast_winner"))

,split,rows,mae,rmse,wape,positive_demand_mae,zero_demand_fraction,baseline
0,test,1836720,0.003237,0.064491,1.000000,1.103788,0.997068,always_zero
1,test,1836720,0.006412,0.089392,1.980992,1.074824,0.997068,last_day
2,test,1836720,0.006965,0.068874,2.151916,1.069970,0.997068,moving_average_7
3,test,1836720,0.007370,0.097563,2.277040,1.090234,0.997068,seasonal_naive_28
4,test,1836720,0.007476,0.097761,2.309840,1.082250,0.997068,weekly_seasonal_naive
5,validation,1836720,0.003943,0.071573,1.000000,1.108849,0.996444,always_zero
6,validation,1836720,0.007379,0.097987,1.871186,1.093539,0.996444,weekly_seasonal_naive
7,validation,1836720,0.007511,0.075472,1.904795,1.085229,0.996444,moving_average_7
8,validation,1836720,0.007559,0.099432,1.916747,1.095224,0.996444,seasonal_naive_28
9,validation,1836720,0.007673,0.099716,1.945741,1.087263,0.996444,last_day


,split,rows,mae,rmse,wape,positive_demand_mae,zero_demand_fraction,baseline
9,test,1836720,0.003237,0.064491,1.0,1.103788,0.997068,always_zero


Selected on validation MAE: always_zero


## 4. Inspect one product/state history

Choose an active series from validation, then inspect its next-day forecasts. The selection is for illustration only; all series above are included in the metrics. Test results are reported for comparison, but baseline selection uses validation only.

In [5]:
example_series = con.sql("""SELECT product_id, customer_state
    FROM forecast_predictions WHERE split = 'validation'
    GROUP BY product_id, customer_state
    ORDER BY SUM(actual) DESC, product_id, customer_state LIMIT 1""").fetchone()
example_forecasts = con.execute("""SELECT purchase_date AS feature_date,
        purchase_date + INTERVAL 1 DAY AS forecast_date, actual,
        last_day, weekly_seasonal_naive, moving_average_7, seasonal_naive_28
    FROM forecast_predictions WHERE product_id = ? AND customer_state = ?
    ORDER BY purchase_date LIMIT 14""", list(example_series)).df()
display(example_forecasts)
example_forecasts.to_csv(OUTPUT / "example_forecasts.csv", index=False)
con.close()

,feature_date,forecast_date,actual,last_day,weekly_seasonal_naive,moving_average_7,seasonal_naive_28
0,2018-07-05,2018-07-06,1.0,0.0,0.0,0.142857,0.0
1,2018-07-06,2018-07-07,0.0,1.0,0.0,0.285714,0.0
2,2018-07-07,2018-07-08,1.0,0.0,0.0,0.285714,0.0
3,2018-07-08,2018-07-09,0.0,1.0,0.0,0.428571,0.0
4,2018-07-09,2018-07-10,0.0,0.0,1.0,0.428571,1.0
5,2018-07-10,2018-07-11,0.0,0.0,0.0,0.285714,0.0
6,2018-07-11,2018-07-12,0.0,0.0,0.0,0.285714,1.0
7,2018-07-12,2018-07-13,0.0,0.0,1.0,0.285714,0.0
8,2018-07-13,2018-07-14,0.0,0.0,0.0,0.142857,0.0
9,2018-07-14,2018-07-15,0.0,0.0,1.0,0.142857,1.0


## 5. Define chronological recommendation evaluation

We create a snapshot at the start of each month, recommend once, then check purchases during that month.

- Validation snapshot: July 1, 2018; labels: July purchases.
- Test snapshot: August 1, 2018; labels: August purchases.
- Features: only orders purchased **and delivered before the snapshot**.
- Labels: purchases within the following month that eventually reached delivered status. These outcomes are used only for evaluation, not to build recommendations.

All customers with at least one delivered purchase in the label period are evaluated, including new customers. This is an **offline purchasing-customer cohort**, not all site visitors. Olist is sparse in repeat customers; we report warm/new cohorts separately.

Repeat purchases are allowed consistently for all baselines. A future product absent from the historical catalog still counts as a relevance miss; we do not silently remove it from the recall denominator.

In [6]:
K = 10
RECENT_DAYS = 90
REC_PERIODS = {
    "validation": (pd.Timestamp("2018-07-01"), pd.Timestamp("2018-08-01")),
    "test": (pd.Timestamp("2018-08-01"), pd.Timestamp("2018-09-01")),
}
assert K > 0
assert REC_PERIODS["validation"][1] <= REC_PERIODS["test"][0]

def load_labels(start, end):
    with duckdb.connect(str(WAREHOUSE), read_only=True) as db:
        return db.execute("""SELECT DISTINCT customer_unique_id, product_id
            FROM order_items
            WHERE order_status = 'delivered'
              AND CAST(order_purchase_timestamp AS TIMESTAMP) >= ?
              AND CAST(order_purchase_timestamp AS TIMESTAMP) < ?
            ORDER BY customer_unique_id, product_id""",
            [start.to_pydatetime(), end.to_pydatetime()]).df()

print("Top K:", K)
display(pd.DataFrame(REC_PERIODS, index=["snapshot", "label_end_exclusive"]).T)

,snapshot,label_end_exclusive
validation,2018-07-01,2018-08-01
test,2018-08-01,2018-09-01


Top K: 10


## 6. Build three simple recommendation rules

1. **Global popularity:** recommend products with the most purchase-item rows in the past 90 days. If the entire recent window is empty, use all history.
2. **Category popularity:** recommend popular products from any category in the customer's recent purchases, then fill remaining positions with global popularity.
3. **Purchase history:** rank the customer's previously purchased products by frequency, breaking ties by recency, then fill with global popularity.

For new customers, category/history rules fall back to global popularity. Lists are deduplicated and ties use product ID for reproducibility. Global recent popularity can return fewer than K products when fewer were purchased in that window.

**Inventory-filtered popularity is skipped:** there are no inventory snapshots in Olist. Historical sales do not establish current availability. Add a dated stock/recall source before implementing that baseline.

In [7]:
def unique_top(products, k):
    return list(dict.fromkeys(products))[:k]

def prepare_recommender(history, cutoff):
    recent = history.loc[history.purchased_at.ge(cutoff - pd.Timedelta(days=RECENT_DAYS))]
    popular_history = recent if not recent.empty else history
    counts = popular_history.groupby("product_id").size().rename("count").reset_index()
    global_order = counts.sort_values(["count", "product_id"], ascending=[False, True]).product_id.tolist()
    categories = history.drop_duplicates("product_id").set_index("product_id").category.to_dict()
    recent_categories = recent.groupby("customer_unique_id").category.agg(set).to_dict()
    preferences = history.groupby(["customer_unique_id", "product_id"], as_index=False).agg(
        frequency=("order_id", "size"), last_purchase=("purchased_at", "max"))
    preferences = preferences.sort_values(
        ["customer_unique_id", "frequency", "last_purchase", "product_id"],
        ascending=[True, False, False, True])
    history_orders = preferences.groupby("customer_unique_id").product_id.agg(list).to_dict()
    # Cache category lists; do not rescan the full catalog for every customer.
    category_lists = {}
    def recommend(customer, method, k):
        if method == "global_popularity":
            selected = global_order
        elif method == "category_popularity":
            key = frozenset(recent_categories.get(customer, set()))
            if key not in category_lists:
                category_lists[key] = [p for p in global_order if categories.get(p) in key][:k]
            selected = category_lists[key] + global_order[:k]
        elif method == "purchase_history":
            selected = history_orders.get(customer, [])[:k] + global_order[:k]
        else:
            raise ValueError(method)
        return unique_top(selected, k)
    return recommend

# A tiny example: both customers' purchases are known before the cutoff.
toy_history = pd.DataFrame({
    "order_id": ["O1", "O2", "O3", "O3", "O4"],
    "customer_unique_id": ["C1", "C2", "C2", "C2", "C3"],
    "product_id": ["Shoes", "Mixer", "Mixer", "Socks", "Bottle"],
    "category": ["Sports", "Kitchen", "Kitchen", "Sports", "Sports"],
    "purchased_at": pd.to_datetime(["2018-06-01"] * 5),
})
toy_recommend = prepare_recommender(toy_history, pd.Timestamp("2018-07-01"))
display(pd.DataFrame({m: toy_recommend("C1", m, 3) for m in
                     ["global_popularity", "category_popularity", "purchase_history"]}))
assert toy_recommend("C1", "global_popularity", 3)[0] == "Mixer"
assert "Mixer" not in toy_recommend("C1", "category_popularity", 3)
assert toy_recommend("C1", "purchase_history", 3)[0] == "Shoes"
assert toy_recommend("NEW", "purchase_history", 3) == toy_recommend("NEW", "global_popularity", 3)

,global_popularity,category_popularity,purchase_history
0,Mixer,Bottle,Shoes
1,Bottle,Shoes,Mixer
2,Shoes,Socks,Bottle


## 7. Score recommendation quality

If we recommend `[Shoes, Socks, Bottle]` and the customer buys `{Socks, Bag}`:

- Precision@3 = 1 / 3.
- Recall@3 = 1 / 2.
- Hit rate = 1 because at least one recommended item was purchased.
- NDCG@3 also considers position: the hit at position 2 gets less credit than one at position 1.

We average customer-level metrics (macro average). Precision uses K as denominator even if a method returns fewer products. Catalog coverage measures how much of the historical catalog is recommended across the cohort. Label-catalog coverage shows how many future relevant items could possibly have been recommended from that catalog.

In [8]:
def ranking_metrics(recommendations, relevant, k):
    recommendations = recommendations[:k]
    hits = [int(p in relevant) for p in recommendations]
    dcg = sum(hit / math.log2(position + 2) for position, hit in enumerate(hits))
    ideal = sum(1 / math.log2(position + 2) for position in range(min(k, len(relevant))))
    return {"precision_at_k": sum(hits) / k,
            "recall_at_k": sum(hits) / len(relevant) if relevant else 0.0,
            "hit_rate_at_k": float(any(hits)),
            "ndcg_at_k": dcg / ideal if ideal else 0.0}

example_metrics = ranking_metrics(["Shoes", "Socks", "Bottle"], {"Socks", "Bag"}, 3)
display(pd.Series(example_metrics, name="Worked example"))
assert np.isclose(example_metrics["precision_at_k"], 1 / 3)
assert example_metrics["recall_at_k"] == 0.5
assert ranking_metrics(["A"], {"A"}, 1)["ndcg_at_k"] == 1.0
assert ranking_metrics(["A"], {"B"}, 1)["ndcg_at_k"] == 0.0

precision_at_k    0.333333
recall_at_k       0.500000
hit_rate_at_k     1.000000
ndcg_at_k         0.386853
Name: Worked example, dtype: float64

In [9]:
REC_METHODS = ["global_popularity", "category_popularity", "purchase_history"]
recommendation_rows = []
cohort_rows = []
example_recommendations = []
for split, (cutoff, end) in REC_PERIODS.items():
    history = load_history(WAREHOUSE, cutoff)
    labels = load_labels(cutoff, end)
    if history.empty or labels.empty:
        raise ValueError(f"No history or outcomes for {split}; choose a different period")
    recommend = prepare_recommender(history, cutoff)
    truths = labels.groupby("customer_unique_id").product_id.agg(set).to_dict()
    known_customers = set(history.customer_unique_id)
    catalog_ids = set(history.product_id)
    label_catalog_coverage = labels.product_id.isin(catalog_ids).mean()
    for method in REC_METHODS:
        records = []
        recommended_catalog = set()
        for customer, relevant in truths.items():
            products = recommend(customer, method, K)
            assert len(products) == len(set(products))
            assert set(products).issubset(catalog_ids)
            recommended_catalog.update(products)
            records.append({**ranking_metrics(products, relevant, K),
                            "cohort": "warm" if customer in known_customers else "new"})
            if customer == next(iter(truths)):
                example_recommendations.append({"split": split, "baseline": method,
                    "recommendations": ", ".join(products), "actual_purchases": ", ".join(sorted(relevant))})
        scores = pd.DataFrame(records)
        overall = scores.drop(columns="cohort").mean().to_dict()
        recommendation_rows.append({"split": split, "baseline": method, **overall,
            "customers": len(scores), "warm_customers": int(scores.cohort.eq("warm").sum()),
            "catalog_coverage": len(recommended_catalog) / len(catalog_ids),
            "label_catalog_coverage": label_catalog_coverage})
        for cohort, group in scores.groupby("cohort"):
            cohort_rows.append({"split": split, "baseline": method, "cohort": cohort,
                "customers": len(group), **group.drop(columns="cohort").mean().to_dict()})
recommendation_metrics = pd.DataFrame(recommendation_rows)
recommendation_cohorts = pd.DataFrame(cohort_rows)
display(recommendation_metrics)
display(recommendation_cohorts)
display(pd.DataFrame(example_recommendations))
recommendation_metrics.to_csv(OUTPUT / "recommendation_metrics.csv", index=False)
recommendation_cohorts.to_csv(OUTPUT / "recommendation_cohorts.csv", index=False)
rec_winner = (recommendation_metrics.query("split == 'validation'")
              .sort_values(["ndcg_at_k", "baseline"], ascending=[False, True]).iloc[0].baseline)
print("Selected on validation NDCG@K:", rec_winner)
display(recommendation_metrics.query("split == 'test' and baseline == @rec_winner"))

,split,baseline,precision_at_k,recall_at_k,hit_rate_at_k,ndcg_at_k,customers,warm_customers,catalog_coverage,label_catalog_coverage
0,validation,global_popularity,0.001197,0.011803,0.011967,0.005722,6100,139,0.000362,0.613826
1,validation,category_popularity,0.001279,0.012623,0.012787,0.006158,6100,139,0.008145,0.613826
2,validation,purchase_history,0.001492,0.014590,0.014918,0.008388,6100,139,0.006154,0.613826
3,test,global_popularity,0.001791,0.017538,0.017908,0.007023,6310,153,0.000339,0.602479
4,test,category_popularity,0.001854,0.018172,0.018542,0.007598,6310,153,0.007532,0.602479
5,test,purchase_history,0.001997,0.019308,0.019810,0.008688,6310,153,0.006751,0.602479


,split,baseline,cohort,customers,precision_at_k,recall_at_k,hit_rate_at_k,ndcg_at_k
0,validation,global_popularity,new,5961,0.001208,0.011995,0.012079,0.005752
1,validation,global_popularity,warm,139,0.000719,0.003597,0.007194,0.004411
2,validation,category_popularity,new,5961,0.001208,0.011995,0.012079,0.005752
3,validation,category_popularity,warm,139,0.004317,0.039568,0.043165,0.023570
4,validation,purchase_history,new,5961,0.001208,0.011995,0.012079,0.005752
5,validation,purchase_history,warm,139,0.013669,0.125899,0.136691,0.121403
6,test,global_popularity,new,6157,0.001787,0.017595,0.017866,0.007033
7,test,global_popularity,warm,153,0.001961,0.015251,0.019608,0.006597
8,test,category_popularity,new,6157,0.001787,0.017595,0.017866,0.007033
9,test,category_popularity,warm,153,0.004575,0.041394,0.045752,0.030329


,split,baseline,recommendations,actual_purchases
0,validation,global_popularity,"53b36df67ebb7c41585e8d54d6772e08, aca2eb7d00ea...","38021cef829efa264df6f9d21c4df6e9, d7a3cfa08519..."
1,validation,category_popularity,"53b36df67ebb7c41585e8d54d6772e08, aca2eb7d00ea...","38021cef829efa264df6f9d21c4df6e9, d7a3cfa08519..."
2,validation,purchase_history,"53b36df67ebb7c41585e8d54d6772e08, aca2eb7d00ea...","38021cef829efa264df6f9d21c4df6e9, d7a3cfa08519..."
3,test,global_popularity,"53b36df67ebb7c41585e8d54d6772e08, aca2eb7d00ea...",e9aa3fd3f316b44c85f671552a04fa4a
4,test,category_popularity,"53b36df67ebb7c41585e8d54d6772e08, aca2eb7d00ea...",e9aa3fd3f316b44c85f671552a04fa4a
5,test,purchase_history,"53b36df67ebb7c41585e8d54d6772e08, aca2eb7d00ea...",e9aa3fd3f316b44c85f671552a04fa4a


,split,baseline,precision_at_k,recall_at_k,hit_rate_at_k,ndcg_at_k,customers,warm_customers,catalog_coverage,label_catalog_coverage
5,test,purchase_history,0.001997,0.019308,0.01981,0.008688,6310,153,0.006751,0.602479


Selected on validation NDCG@K: purchase_history


## 8. Record baseline runs in MLflow

Each rule gets its own run, with validation and test metrics, dates, parameters and CSV artifacts. Selection is based on validation MAE for forecasting and validation NDCG@K for recommendations. We record all test results for transparent comparison; do not tune settings using them.

The notebook records metrics, not serialized models, since these rules have no fitted model object. Rerunning creates new MLflow runs and overwrites the CSV summaries in `data/baselines/`.

In [10]:
run_rows = []
for task, table in [("forecasting", forecast_metrics), ("recommendations", recommendation_metrics)]:
    for method in table.baseline.unique():
        with mlflow.start_run(run_name=f"{task}__{method}") as run:
            mlflow.log_params({"task": task, "baseline": method,
                               "evaluation": "rolling_next_day" if task == "forecasting" else "monthly_snapshot"})
            mlflow.set_tag("selected_on_validation", str(method == (forecast_winner if task == "forecasting" else rec_winner)))
            if task == "forecasting":
                mlflow.log_params({"validation_feature_start": ranges["validation"]["min"],
                    "validation_feature_end": ranges["validation"]["max"],
                    "test_feature_start": ranges["test"]["min"], "test_feature_end": ranges["test"]["max"],
                    "target_horizon_days": 1,
                    "season_days": 7 if method == "weekly_seasonal_naive" else 28 if method == "seasonal_naive_28" else 0,
                    "average_window_days": 7 if method == "moving_average_7" else 0})
            else:
                mlflow.log_params({"k": K, "popularity_window_days": RECENT_DAYS,
                    "repeat_products_allowed": True, "inventory_filter": "unavailable_in_olist",
                    **{f"{s}_{bound}": str(d) for s, dates in REC_PERIODS.items()
                       for bound, d in zip(["snapshot", "label_end"], dates)}})
            subset = table.loc[table.baseline.eq(method)]
            for _, row in subset.iterrows():
                for name, value in row.items():
                    if name not in ("split", "baseline") and isinstance(value, (int, float, np.number)) and np.isfinite(value):
                        mlflow.log_metric(f"{row['split']}_{name}", float(value))
            artifact = OUTPUT / f"{task}_{method}.csv"
            subset.to_csv(artifact, index=False)
            mlflow.log_artifact(str(artifact))
            if task == "recommendations":
                mlflow.log_artifact(str(OUTPUT / "recommendation_cohorts.csv"))
            run_rows.append({"task": task, "baseline": method, "run_id": run.info.run_id})
display(pd.DataFrame(run_rows))
print("Saved summaries to", OUTPUT)
print("Open MLflow from the repository root with:")
print(".venv/bin/mlflow ui --backend-store-uri", mlflow.get_tracking_uri())

,task,baseline,run_id
0,forecasting,last_day,7e6385aa4d9a44e29d02ff51daab0a5f
1,forecasting,weekly_seasonal_naive,bfce9ded022d4362b0b21519b6932520
2,forecasting,moving_average_7,4c4b7eed04e14aa684611b4c1cfa80cb
3,forecasting,seasonal_naive_28,3f0516ff6e704e4e8782a60f58a331ce
4,forecasting,always_zero,1e2f2421a79f428a9431df68388dd37b
5,recommendations,global_popularity,920851fe1d6c4e8d8ed1ba355ef05f7f
6,recommendations,category_popularity,a8b8909406ab4385939952630e542611
7,recommendations,purchase_history,c9d45d9eed3f429da7ce2cc56be7704f


Saved summaries to /Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/data/baselines
Open MLflow from the repository root with:
.venv/bin/mlflow ui --backend-store-uri sqlite:////Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/data/baselines/mlflow.db


## 9. Interpret the results

- Lower forecasting MAE/RMSE is better. Check positive-demand MAE and the zero fraction before trusting a low overall error.
- Higher recommendation Recall@K/NDCG@K is better. Compare identical periods, customer cohorts, K and eligibility rules.
- A small or zero recommendation metric can be real: Olist has few repeat customers, many products, and substantial new-customer activity. Do not add future purchases to the features to improve the result.
- Category and history rules can match global popularity when customers have no known history. The warm-cohort table helps reveal this.
- The 28-day seasonal rule is a comparison hypothesis, not evidence that demand really repeats every 28 days.
- These evaluations describe historical prediction, not causal sales lift. Baselines have no inventory checks, deployment integration or production approval gate.

A future trained model should improve on the relevant baseline under the same protocol. Keep the baseline independently callable for fallback and comparison.

### Results from the local dataset

The executed example has about 99.64% zero targets in validation and 99.71% in test.
The always-zero diagnostic beats the other forecasting rules on overall MAE, so it
is included in baseline selection rather than hidden. It still misses every positive
sale; review positive-demand error and business costs before choosing a production rule.
Purchase-history recommendations with global fallback have the highest validation
NDCG@10 of the three recommendation rules in this run. Only 139/6,100 validation
customers and 153/6,310 test customers have known purchase history at their cutoffs.
These are observed results, not general claims that one method always wins.

Verification: all Python cells were run sequentially in the project environment;
a lightweight display renderer was substituted because that environment has no
IPython/Jupyter kernel installed. DataFrame outputs below are saved in the notebook.
To run interactively, install `ipykernel` in the project environment and select it
as the notebook kernel. No kernel dependency was installed by this change.
